# Multipart MSD Complexity-Error Audit

This notebook is the workflow for a no-training diagnostic of SuSuInterActs and the frozen multipart Step 2 infiller. Reusable loading, inference, statistics, caching, and plotting functions live in `utils/msd/complexity_error_audit.py`.

The audit asks whether model error is associated with physical speed, spectral energy, or MSD spectral spread, globally and for upper body, lower body, feet, hands, and RVQ levels. Ground-truth complexity is used only for post-hoc evaluation, so it does not leak into inference.

## 1. Imports and project root

Run the notebook from any directory inside the repository.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
while not (PROJECT_DIR / 'motion_generation').is_dir() and PROJECT_DIR != PROJECT_DIR.parent:
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / 'motion_generation').is_dir():
    raise RuntimeError('Repository root not found')
for path in (PROJECT_DIR, PROJECT_DIR / 'motion_generation'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from motion_generation.utils.msd.complexity_error_audit import (
    add_empirical_strata,
    correlation_summary,
    decoded_error_by_complexity,
    default_audit_config,
    distribution_summary,
    error_by_complexity,
    kruskal_summary,
    path_status,
    plot_clip_timeline,
    plot_complexity_distributions,
    plot_error_quartiles,
    plot_part_omega_correlation,
    plot_speed_complexity_joint,
    plot_speed_distributions,
    quartile_effect_summary,
    regime_summary,
    run_or_load_audit,
    select_example_clips,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
print('PROJECT_DIR:', PROJECT_DIR)

## 2. Experiment configuration

The default is a complete 64-clip pilot, including FK speed and decoded errors. Once it passes, set `MAX_CLIPS = None` and use a new output directory for the full validation census. Set the checkpoint path to the final multipart bidirectional Step 2 directory containing `config.json` and model weights.

In [ ]:
RUN_EXTRACTION = True
OVERWRITE_CACHE = False
MAX_CLIPS = 64             # None for all validation clips
MAX_WINDOWS = None         # None uses every non-overlapping Step 2 window
INCLUDE_FK_SPEED = True
INCLUDE_DECODED_ERRORS = True
BOOTSTRAP_ITERATIONS = 500

config = default_audit_config(PROJECT_DIR)
config.device = os.environ.get('MSD_AUDIT_DEVICE', 'cuda:0')
config.mask_checkpoint = Path(os.environ.get('MP_STEP2_CKPT', config.mask_checkpoint))
config.max_clips = MAX_CLIPS
config.max_windows = MAX_WINDOWS
config.batch_size = 128
config.generate_steps = 1
config.include_fk_speed = INCLUDE_FK_SPEED
config.include_decoded_errors = INCLUDE_DECODED_ERRORS
config.output_dir = PROJECT_DIR / 'motion_generation' / 'outputs' / (
    'multipart_complexity_error_audit_full' if MAX_CLIPS is None else f'multipart_complexity_error_audit_pilot_{MAX_CLIPS}'
)

display(path_status(config))
print('Cache/output:', config.output_dir)

## 3. Extract or load cached records

The first run loads the frozen infiller and four codecs, evaluates all selected windows, and writes compressed tables. Later runs load those tables unless `OVERWRITE_CACHE=True`. The cache contains per-target-frame, per-token, per-clip, failure, and metadata files.

In [ ]:
tables, context = run_or_load_audit(
    config,
    run_extraction=RUN_EXTRACTION,
    overwrite=OVERWRITE_CACHE,
)
tables = add_empirical_strata(tables)

print('frames:', len(tables.frames))
print('tokens:', len(tables.tokens))
print('clips:', tables.frames['name'].nunique())
print('failures:', len(tables.failures))
display(tables.failures.head(20))

## 4. Coverage and empirical distributions

`tau_noise` is schema-specific FK speed. Maya does not have a validated `tau_active`, so moving frames are divided into empirical speed and MSD quartiles. Chonglu's provisional active threshold remains only a diagnostic overlay.

In [ ]:
display(regime_summary(tables.frames).round(5))
display(distribution_summary(tables.frames).round(6))

In [ ]:
fig = plot_speed_distributions(tables.frames)
plt.show()

fig = plot_complexity_distributions(tables.frames)
plt.show()

fig = plot_speed_complexity_joint(tables.frames)
plt.show()

## 5. Does infill error increase with complexity?

Teacher-forced NLL is the cleanest continuous error measure. Generated accuracy measures the actual one-shot or iterative fill path. Results are separated by part and RVQ level because a combined average can hide hand or residual-code failures.

In [ ]:
complexity_errors = error_by_complexity(tables.tokens)
display(complexity_errors.round(5))

fig = plot_error_quartiles(tables.tokens, metric='teacher_nll')
plt.show()
fig = plot_error_quartiles(tables.tokens, metric='generated_correct')
plt.show()

## 6. Statistical association and effect size

Spearman confidence intervals resample complete clips, not individual frames. The quartile table reports the NLL change and Cliff's delta from Q1 to Q4. Kruskal-Wallis is included as a descriptive omnibus test; with many frames, effect size and clip-bootstrap intervals matter more than a small p-value.

In [ ]:
correlations = correlation_summary(
    tables.tokens,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=config.seed,
)
display(correlations.round(5))
display(quartile_effect_summary(tables.tokens).round(5))
display(kruskal_summary(tables.tokens))

## 7. Decoded motion errors

When `INCLUDE_DECODED_ERRORS=True`, this section reports feature MAE/RMSE, velocity and acceleration RMSE, and 6D rotation geodesic error for combined motion and every part. Each five-token window is decoded with both anchors present before middle-frame errors are measured.

In [ ]:
decoded_summary = decoded_error_by_complexity(tables.frames)
if decoded_summary.empty:
    print('Decoded columns are absent. Re-run with INCLUDE_DECODED_ERRORS=True and a new cache.')
else:
    display(decoded_summary.round(6))

## 8. Part coupling and representative timelines

The correlation map tests whether a single global complexity score is likely to hide independent part dynamics. Timelines provide a qualitative check that high MSD corresponds to meaningful irregular motion rather than isolated token jitter.

In [ ]:
fig = plot_part_omega_correlation(tables.frames)
plt.show()

example_clips = select_example_clips(tables.frames, count_per_tier=2)
display(example_clips.round(5))

In [ ]:
for clip_name in example_clips['name'].drop_duplicates().head(6):
    fig = plot_clip_timeline(tables.frames, clip_name)
    plt.show()

## 9. Interpretation gates

Use the results to choose the next training experiment:

- **Part Omega correlates with NLL after controlling through comparisons with speed and energy:** test part-specific MSD weighting or an auxiliary spectral target.
- **Speed or energy correlates, but Omega does not:** use a simpler magnitude/velocity baseline; MSD is not yet justified.
- **Only hands or later quantizers degrade in Q4:** target those slots instead of globally reweighting every token.
- **Token error rises but decoded error does not:** exact token accuracy is overstating a perceptually harmless code substitution.
- **Decoded error rises with Q4 and the effect survives clip bootstrap:** this is the strongest evidence for an MSD-based transformer experiment.

For the final census, set `MAX_CLIPS=None`, keep `MAX_WINDOWS=None`, choose a fresh full-run output directory, and rerun from configuration. Do not compare pilot p-values as final evidence.